# Cross-asset HDFC ⇄ Nifty
- **beta_log**: rolling \( \mathrm{Cov}(r_h,r_n)/\mathrm{Var}(r_n) \) on overlapping **daily** log returns (`r = Δ ln(close)`).
- **Windows**: "60‑day beta" uses the trailing **60** return observations ⇒ need roughly **61** aligned closes.
- Data: Yahoo (`HDFCBANK.NS`, `^NSEI`). Swap loaders if you use `nselib`.

In [2]:
import numpy as np
import pandas as pd
import yfinance as yf


def fetch_aligned_closes(symbol_h="HDFCBANK.NS", symbol_n="^NSEI", period="2y"):
    tickers = [symbol_h, symbol_n]
    data = yf.download(
        tickers,
        period=period,
        interval="1d",
        auto_adjust=True,
        progress=False,
        threads=True,
    )
    if isinstance(data.columns, pd.MultiIndex):
        close = data["Close"].copy()
    else:
        close = data[["Close"]].copy()
        close.columns = tickers
    px = (
        pd.DataFrame({"h": close[symbol_h], "n": close[symbol_n]})
        .dropna()
        .sort_index()
    )
    lh = np.log(px["h"]).diff()
    ln = np.log(px["n"]).diff()
    ret = pd.DataFrame({"lh": lh, "ln": ln}).dropna()
    return px, ret


def rolling_beta_log(lh: pd.Series, ln: pd.Series, window: int) -> pd.Series:
    """Cov(lh, ln) / Var(ln) over a trailing window of overlapping daily log returns (ddof=1)."""
    cov = lh.rolling(window=window, min_periods=window).cov(ln)
    var_n = ln.rolling(window=window, min_periods=window).var(ddof=1)
    return cov / var_n


_px, _rets = fetch_aligned_closes()
_beta60 = rolling_beta_log(_rets["lh"], _rets["ln"], 60)
_beta90 = rolling_beta_log(_rets["lh"], _rets["ln"], 90)

_summary = pd.DataFrame(
    {
        "latest_date": [
            _beta60.dropna().index[-1],
            _beta90.dropna().index[-1],
        ],
        "beta_log": [
            float(_beta60.dropna().iloc[-1]),
            float(_beta90.dropna().iloc[-1]),
        ],
    },
    index=["60d", "90d"],
)
display(_summary)

betas = pd.DataFrame({"beta_log_60d": _beta60, "beta_log_90d": _beta90}).dropna(how="all")
betas.tail()

,latest_date,beta_log
60d,2026-05-15,1.379384
90d,2026-05-15,1.278455


,beta_log_60d,beta_log_90d
Date,,
2026-05-11,1.365929,1.274441
2026-05-12,1.359471,1.262820
2026-05-13,1.362172,1.262818
2026-05-14,1.378684,1.278630
2026-05-15,1.379384,1.278455


In [3]:
"""
Cross-asset sensitivity: how HDFC Bank option delta changes per unit move in Nifty.

Interpretation of d_delta_d_nifty:
  - If Nifty spot increases by 1 (same currency as S_nifty), delta changes by about d_delta_d_nifty.
  - Scale: multiply by expected Nifty move in points to get expected delta change.
"""

from scipy.stats import norm


def bs_d1_d2(S, K, T, r, q, sigma):
    if T <= 0 or sigma <= 0:
        raise ValueError("Need T > 0 and sigma > 0")
    vsqrt = sigma * np.sqrt(T)
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / vsqrt
    d2 = d1 - vsqrt
    return d1, d2


def bs_call_delta_gamma_vanna(S, K, T, r, q, sigma):
    """Returns delta, gamma, vanna (d delta / d sigma), for a European call."""
    d1, d2 = bs_d1_d2(S, K, T, r, q, sigma)
    disc = np.exp(-q * T)
    delta = disc * norm.cdf(d1)
    gamma = disc * norm.pdf(d1) / (S * sigma * np.sqrt(T))
    vanna = -disc * norm.pdf(d1) * d2 / sigma
    return delta, gamma, vanna


def d_sh_per_d_sn_from_log_beta(S_hdfc, S_nifty, beta_log):
    """beta_log = Cov(ln ret_h, ln ret_n) / Var(ln ret_n); maps dS_nifty -> dS_hdfc."""
    return beta_log * (S_hdfc / S_nifty)


def cross_delta_gamma_to_nifty(
    S_hdfc,
    K,
    T,
    r,
    q,
    sigma,
    S_nifty,
    *,
    beta_log=None,
    d_sh_d_sn=None,
    dsigma_per_d_snifty=0.0,
    option_type="call",
):
    if beta_log is not None and d_sh_d_sn is not None:
        raise ValueError("Pass only one of beta_log or d_sh_d_sn")
    if beta_log is not None:
        d_sh_d_sn = d_sh_per_d_sn_from_log_beta(S_hdfc, S_nifty, beta_log)
    if d_sh_d_sn is None:
        raise ValueError("Need beta_log or d_sh_d_sn")

    sign = 1.0 if option_type == "call" else -1.0
    d1, d2 = bs_d1_d2(S_hdfc, K, T, r, q, sigma)
    disc = np.exp(-q * T)
    gamma = disc * norm.pdf(d1) / (S_hdfc * sigma * np.sqrt(T))
    vanna = -disc * norm.pdf(d1) * d2 / sigma

    out = {
        "gamma": gamma,
        "vanna": vanna,
        "d_sh_d_sn": d_sh_d_sn,
        "d_delta_call_d_snifty": sign * gamma * d_sh_d_sn + sign * vanna * dsigma_per_d_snifty,
    }
    if option_type == "put":
        delta_put = -disc * norm.cdf(-d1)
        out["delta_opt"] = delta_put
        out["d_delta_put_d_snifty"] = -gamma * d_sh_d_sn - vanna * dsigma_per_d_snifty
    else:
        delta_call, _, _ = bs_call_delta_gamma_vanna(S_hdfc, K, T, r, q, sigma)
        out["delta_opt"] = delta_call
    return out


# Example: plug latest beta_log from prior cell (`_beta60`, `_beta90`)
Sn = float(_px["n"].iloc[-1])
Sh = float(_px["h"].iloc[-1])
bl = float(_beta60.dropna().iloc[-1])
res = cross_delta_gamma_to_nifty(
    Sh,
    Sh,
    20 / 365.0,
    0.06,
    0.0,
    0.22,
    Sn,
    beta_log=bl,
    option_type="call",
)
print("Using 60d beta_log:", bl)
print(res)

Using 60d beta_log: 1.3793837895643901
{'gamma': np.float64(0.010053043090502001), 'vanna': np.float64(-0.06879761854948399), 'd_sh_d_sn': 0.04477666413562583, 'd_delta_call_d_snifty': np.float64(0.000450141734004382), 'delta_opt': np.float64(0.5356933564584938)}
